# Dispersion Practical (ABA Unit 1, Practical 4)

**Aim:** calculate variance and standard deviation for a business dataset (Sahyadri Mart daily sales, synthetic).

Run the cells in order (Runtime → Run all). Nothing to install. The note `dispersion-practical` on Sutra explains every step.

In [1]:
# Step 1. Load the data and pick one store: S05, the Pune Express store
import pandas as pd
import numpy as np

url = ("https://raw.githubusercontent.com/Prathameshv07/ai-ds-notes/main/"
       "content/assets/data/sahyadri-mart-sales.csv")
df = pd.read_csv(url, parse_dates=["date"])
s05 = df[df["store_id"] == "S05"]
print("Rows, columns:", df.shape, "| days for S05:", len(s05))

Rows, columns: (8772, 16) | days for S05: 731


In [2]:
# Step 2. Variance and standard deviation by hand: S05 footfall, first 7 days
week = s05["footfall"].head(7).to_numpy()
mean = week.mean()
dev = week - mean
print(pd.DataFrame({"x": week, "x - mean": dev.round(2), "(x - mean)^2": (dev**2).round(2)}).to_string(index=False))
n, ss = len(week), (dev**2).sum()
print(f"\nn = {n}, mean = {mean:.2f}, sum of squared deviations = {ss:.2f}")
print(f"Sample variance     s^2 = SS/(n-1) = {ss/(n-1):.2f}  ->  s     = {np.sqrt(ss/(n-1)):.2f}")
print(f"Population variance     = SS/n     = {ss/n:.2f}  ->  sigma = {np.sqrt(ss/n):.2f}")

   x  x - mean  (x - mean)^2
1036    -73.71       5433.80
 943   -166.71      27793.65
 974   -135.71      18418.37
1088    -21.71        471.51
1108     -1.71          2.94
1569    459.29     210943.37
1050    -59.71       3565.80

n = 7, mean = 1109.71, sum of squared deviations = 266629.43
Sample variance     s^2 = SS/(n-1) = 44438.24  ->  s     = 210.80
Population variance     = SS/n     = 38089.92  ->  sigma = 195.17


In [3]:
# Step 3. The same with pandas and NumPy: watch the default divisor
w = s05["footfall"].head(7)
print("pandas w.var(), w.std()  :", round(w.var(), 2), round(w.std(), 2), "  (divides by n-1)")
print("NumPy  np.var(), np.std():", round(np.var(week), 2), round(np.std(week), 2), "  (divides by n)")
print("NumPy  np.std(ddof=1)    :", round(np.std(week, ddof=1), 2))

pandas w.var(), w.std()  : 44438.24 210.8   (divides by n-1)
NumPy  np.var(), np.std(): 38089.92 195.17   (divides by n)
NumPy  np.std(ddof=1)    : 210.8


In [4]:
# Step 4. All the measures for S05's daily footfall over two years
f = s05["footfall"]
print(f"Mean           : {f.mean():.1f}")
print(f"Range          : {f.max()} - {f.min()} = {f.max() - f.min()}")
print(f"Variance (s^2) : {f.var():.1f}")
print(f"Std dev (s)    : {f.std():.1f}")
print(f"CV             : {f.std() / f.mean() * 100:.1f}%")

Mean           : 1257.3
Range          : 2486 - 721 = 1765
Variance (s^2) : 61607.7
Std dev (s)    : 248.2
CV             : 19.7%


In [5]:
# Step 5. Which kind of store has steadier daily revenue? Compare SD and CV
g = df.groupby("store_type")["revenue_inr"].agg(["mean", "std"])
g["cv_pct"] = g["std"] / g["mean"] * 100
print(g.round(1).to_string())

                 mean       std  cv_pct
store_type                             
Express      213056.1   94144.6    44.2
Supermarket  767425.5  282443.8    36.8


In [6]:
# Step 6. Rank all 12 stores by the CV of daily footfall
s = df.groupby(["store_id", "city"])["footfall"].agg(["mean", "std"])
s["cv_pct"] = s["std"] / s["mean"] * 100
print(s.round(1).sort_values("cv_pct").to_string())

                                      mean    std  cv_pct
store_id city                                            
S06      Kolhapur                    829.7  163.8    19.7
S05      Pune                       1257.3  248.2    19.7
S09      Chhatrapati Sambhajinagar  1407.8  279.1    19.8
S10      Nanded                      699.3  139.8    20.0
S08      Jalgaon                     749.9  151.9    20.3
S02      Thane                      1322.4  269.2    20.4
S01      Mumbai                     2208.6  451.8    20.5
S04      Pune                       2068.4  424.1    20.5
S12      Amravati                    773.2  161.8    20.9
S11      Nagpur                     1628.0  341.4    21.0
S07      Nashik                     1490.6  314.2    21.1
S03      Ratnagiri                   703.0  154.7    22.0


In [7]:
# Step 7. Why the range is fragile: drop S01's single busiest day
r = df.loc[df["store_id"] == "S01", "revenue_inr"]
r2 = r.drop(r.idxmax())
print(f"All 731 days : range Rs {r.max() - r.min():,} | SD Rs {r.std():,.0f}")
print(f"Without 1 day: range Rs {r2.max() - r2.min():,} | SD Rs {r2.std():,.0f}")

All 731 days : range Rs 2,413,570 | SD Rs 250,614
Without 1 day: range Rs 1,623,260 | SD Rs 240,391
